# Lab: Naive Bayes

Companion notebook for the lecture *Probabilistic Model* (`slides/main.tex`).

**How to use this notebook**
- Lines marked `# TODO` are for you: replace `...` with your code.
- Each TODO cell ends with `check(...)`. ✅ means your result matches the slides; ❌ means try again.
- **🧪 Practice** sections are new problems that are not in the slides.
- Work from top to bottom: later cells use variables from earlier cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, beta
from scipy.special import logsumexp

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
rng = np.random.default_rng(0)          # fixed seed: everyone gets the same random numbers


def check(name, value, expected, tol=1e-3):
    """Print ✅ if value ≈ expected (element-wise), otherwise ❌."""
    try:
        ok = np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol)
    except (TypeError, ValueError):
        ok = False
    print(("✅ " if ok else "❌ ") + name)

---
# Part 1 — Learning from Data: MLE
*Slides: Example: Coin Flipping · Coin Flip: Computing Likelihoods · MLE for a Bernoulli Parameter*

Two candidate coins: **straight** $\theta = p(H) = \tfrac12$ and **bent** $\theta = \tfrac45$.
Observed data $D$ = `HTHHHTHHTHTH`.

For i.i.d. flips the likelihood is $\;p(D\mid\theta) = \theta^{\#H}\,(1-\theta)^{\#T}$.

### 1.1 Count the data

In [ ]:
flips = "HTHHHTHHTHTH"
n_heads = ...  # TODO: count the "H" characters
n_tails = ...  # TODO: count the "T" characters
print(f"{n_heads} heads, {n_tails} tails")
check("n_heads", n_heads, 8)
check("n_tails", n_tails, 4)

### 1.2 Likelihood of each coin

In [ ]:
def likelihood(theta, n_heads, n_tails):
    """p(D | theta) for i.i.d. Bernoulli flips."""
    return ...  # TODO: theta^#H * (1 - theta)^#T


lik_straight = likelihood(0.5, n_heads, n_tails)
lik_bent = likelihood(0.8, n_heads, n_tails)
print(f"p(D | straight) = {lik_straight:.3e}")
print(f"p(D | bent)     = {lik_bent:.3e}")
print(f"The data are {lik_bent / lik_straight:.2f}x more probable under the bent coin.")
check("p(D | straight)", lik_straight, 2.44e-4, tol=1e-6)
check("p(D | bent)", lik_bent, 2.68e-4, tol=1e-6)

> ⚠️ **Careful with the wording** (slide *Coin Flip: Computing Likelihoods*):
> this compares $p(D\mid\text{coin})$, **not** $p(\text{coin}\mid D)$.
> It does *not* say the bent coin is more likely to be the coin we are holding.

### 1.3 MLE when $\theta$ can be any value in $[0, 1]$
Evaluate $L(\theta)$ on a fine grid and pick the $\theta$ with the largest value.
Compare it with the closed form $\hat\theta = \#H / N$.

In [ ]:
thetas = np.linspace(0, 1, 1001)
L = likelihood(thetas, n_heads, n_tails)      # our function also works on arrays
theta_hat = ...  # TODO: the theta on the grid where L is largest (hint: np.argmax)
print(f"grid MLE = {theta_hat:.3f},   count / total = {n_heads / (n_heads + n_tails):.3f}")
check("theta_hat", theta_hat, 2 / 3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(thetas, L)
ax[0].axvline(theta_hat, ls="--", c="gray")
ax[0].scatter([0.5, 0.8], [lik_straight, lik_bent], c=["tab:blue", "tab:red"], zorder=3)
ax[0].annotate("straight", (0.5, lik_straight), textcoords="offset points", xytext=(-55, 0))
ax[0].annotate("bent", (0.8, lik_bent), textcoords="offset points", xytext=(8, 0))
ax[0].set(xlabel=r"$\theta$", ylabel=r"$L(\theta)$", title="Likelihood")
with np.errstate(divide="ignore"):
    ax[1].plot(thetas, np.log(L))
ax[1].axvline(theta_hat, ls="--", c="gray")
ax[1].set(xlabel=r"$\theta$", ylabel=r"$\log L(\theta)$", ylim=(-30, 0),
          title="Log-likelihood (same argmax)")
plt.tight_layout()
plt.show()

### 1.4 Problem with MLE: little data
*Slide: Two Problems with MLE.* Flip a coin 3 times and get `HHH`.

In [ ]:
theta_hhh = ...  # TODO: MLE = count / total
print(f"MLE after HHH: p(H) = {theta_hhh},  p(T) = {1 - theta_hhh}")
check("MLE after HHH", theta_hhh, 1.0)

**Q1.** After only three flips, is it reasonable to say that tails is *impossible*?

> **Your answer:** _..._

### 🧪 Practice 1 — Free throws
A basketball player makes **7 of 10** free throws.
Coach A thinks the player's success rate is $\theta = 0.5$; Coach B thinks $\theta = 0.75$.

1. Compute the likelihood of the data under each coach's $\theta$. Which one does MLE choose?
2. What is the MLE $\hat\theta$ if $\theta$ can be any value in $[0, 1]$?

In [ ]:
made, missed = 7, 3
lik_A = ...  # TODO: Coach A
lik_B = ...  # TODO: Coach B
theta_hat_ft = ...  # TODO: MLE for any theta
print(f"p(D | 0.5) = {lik_A:.3e},  p(D | 0.75) = {lik_B:.3e},  theta_hat = {theta_hat_ft}")
check("p(D | theta=0.5)", lik_A, 9.766e-4, tol=1e-6)
check("p(D | theta=0.75)", lik_B, 2.086e-3, tol=1e-6)
check("theta_hat", theta_hat_ft, 0.7)

**Q2.** Which coach does MLE choose, and is the evidence strong?

> **Your answer:** _..._

---
# Part 2 — Bayes' Rule for Decisions
*Slides: Bayes' Rule · Back to the Coin · Computing the Posterior · How Much Does the Prior Matter?*

$$
\underbrace{p(h\mid D)}_{\text{posterior}} = \frac{p(D\mid h)\,p(h)}{p(D)}, \qquad p(D) = \sum_{h'} p(D\mid h')\,p(h')
$$

You will write **one** function `bayes()` here and reuse it for coins, patients, emails and documents.

In [ ]:
def bayes(prior, lik):
    """Posterior over hypotheses: (likelihood × prior) / evidence."""
    prior = np.asarray(prior, dtype=float)
    lik = np.asarray(lik, dtype=float)
    joint = ...  # TODO: step 2: multiply each likelihood by its prior
    evidence = ...  # TODO: step 3: p(D) = sum over hypotheses
    return ...  # TODO: step 4: divide by the evidence


# the coin came from a mint, so it is probably straight
prior_coin = [0.9, 0.1]                     # [straight, bent]
lik_coin = [lik_straight, lik_bent]
post_coin = bayes(prior_coin, lik_coin)
print(f"p(D) = {np.dot(prior_coin, lik_coin):.4e}")
print(f"p(straight | D) = {post_coin[0]:.3f},   p(bent | D) = {post_coin[1]:.3f}")
check("posterior", post_coin, [0.891, 0.109])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, values, title in zip(axes, [prior_coin, lik_coin, post_coin],
                             ["Prior p(h)", "Likelihood p(D | h)", "Posterior p(h | D)"]):
    ax.bar(["straight", "bent"], values, color=["tab:blue", "tab:red"])
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 2.1 How much does the prior matter? (reproduce the table on the slide)

In [ ]:
rows = []
for pr in ([0.5, 0.5], [0.9, 0.1]):
    post = ...  # TODO: posterior for this prior
    rows.append({"prior (straight, bent)": pr, "posterior straight": post[0],
                 "posterior bent": post[1], "decision": ["straight", "bent"][np.argmax(post)]})
prior_table = pd.DataFrame(rows)
display(prior_table)
check("uniform prior", prior_table.loc[0, "posterior straight"], 0.476)
check("mint prior", prior_table.loc[1, "posterior straight"], 0.891)

**Q3.** With the uniform prior (0.5, 0.5) the decision is the same as MLE. Why?

> **Your answer:** _..._

### 2.2 With more data the likelihood wins
Suppose we keep flipping and see the same pattern $k$ times ($8k$ heads, $4k$ tails).
Keep the mint prior (0.9, 0.1). After how many repetitions does the posterior favour the bent coin?

In [ ]:
ks = np.arange(1, 41)
post_bent = []
for k in ks:
    lik_k = ...  # TODO: [p(D|straight), p(D|bent)] with 8k heads, 4k tails
    post_bent.append(bayes([0.9, 0.1], lik_k)[1])
post_bent = np.array(post_bent)

k_switch = ks[np.argmax(post_bent > 0.5)]
print(f"The posterior first favours the bent coin at k = {k_switch} ({12 * k_switch} flips)")
check("k_switch", k_switch, 24)

plt.figure(figsize=(6, 3))
plt.plot(12 * ks, post_bent, marker=".")
plt.axhline(0.5, ls="--", c="gray")
plt.xlabel("number of flips")
plt.ylabel("p(bent | D)")
plt.title("Prior 0.9 / 0.1: the data slowly win")
plt.show()

> 💡 Try `ks = np.arange(1, 201)`. What goes wrong at large $k$? (We fix it in Part 3.8.)

### 🧪 Practice 2 — A medical test
A disease affects **1%** of the population. A test has sensitivity $p(+\mid\text{disease}) = 0.95$
and false-positive rate $p(+\mid\text{no disease}) = 0.10$.

1. A patient tests positive. What is $p(\text{disease}\mid +)$?
2. The patient takes a second, independent test and is positive again. Update your belief
   (today's prior = yesterday's posterior).

In [ ]:
prior_dz = [0.01, 0.99]                 # [disease, no disease]
lik_pos = [0.95, 0.10]                  # [p(+ | disease), p(+ | no disease)]
post_1 = ...  # TODO: after one positive test
post_2 = ...  # TODO: after a second positive test
print(f"p(disease | +)    = {post_1[0]:.3f}")
print(f"p(disease | +, +) = {post_2[0]:.3f}")
check("after 1 test", post_1[0], 0.0876)
check("after 2 tests", post_2[0], 0.4769)

**Q4.** The test is 95% sensitive, yet $p(\text{disease}\mid +)$ is below 10%. Why?

> **Your answer:** _..._

**Q5.** Updating twice with the same likelihood assumes something about the two test results. What? Where will this assumption appear again?

> **Your answer:** _..._

---
# Part 3 — Bernoulli Naive Bayes
*Slides: Example: Problem Setup · Training Counts · Estimate the Class-Conditionals · Classify a New Patient*

Predict disease ($Y=1$) from three binary features: Fever, Cough, Fatigue.
Model: $X_i \mid Y=c \sim \text{Bernoulli}(\theta_{i|c})$ and
$\hat y = \arg\max_c\; p(Y=c)\prod_i p(x_i\mid Y=c)$.

In [ ]:
df_train = pd.DataFrame({
    "Disease": [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    "Fever":   [1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0],
    "Cough":   [1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0],
    "Fatigue": [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
})
features = ["Fever", "Cough", "Fatigue"]
df_train

### 3.1 Training counts (the table on the slide)

In [ ]:
N_c = ...  # TODO: number of patients in each class
counts = ...  # TODO: count(X_i = 1, Y = c) for every feature
display(pd.concat([N_c.rename("N_c"), counts], axis=1).T)
check("N_c", N_c, [7, 5])
check("counts", counts, [[2, 5, 2], [4, 3, 5]])

### 3.2 MLE: count / total
$\hat p(Y=c) = N_c / N \qquad \hat\theta_{i|c} = \text{count}(X_i=1, Y=c) / N_c$

Row 0 = no disease, row 1 = disease (sorted by label, the same order scikit-learn uses).

In [ ]:
prior_y = ...  # TODO: p(Y = c)
theta = ...  # TODO: p(X_i = 1 | Y = c)   hint: divide each row by its N_c with .div(..., axis=0)
display(theta)
check("prior", prior_y, [7 / 12, 5 / 12])
check("theta", theta, [[2 / 7, 5 / 7, 2 / 7], [0.8, 0.6, 1.0]])

> 💡 Shortcut: `df_train.groupby("Disease")[features].mean()` gives the same table, because the mean of 0/1 values is the proportion of 1s.

### 3.3 Classify a new patient $x = (1, 0, 1)$
$p(X_i = x_i\mid Y=c) = \theta_{i|c}$ if $x_i = 1$, and $1-\theta_{i|c}$ if $x_i = 0$.

In [ ]:
def bernoulli_likelihood(x, theta):
    """p(x | Y=c) = prod_i p(X_i = x_i | Y=c), one value per class."""
    x = np.asarray(x)
    p = ...  # TODO: theta where x_i = 1, otherwise 1 - theta
    return ...  # TODO: multiply over the features (the naive assumption)


x_new = [1, 0, 1]                                   # Fever=1, Cough=0, Fatigue=1
lik_x = bernoulli_likelihood(x_new, theta)
s_x = ...  # TODO: unnormalized scores s_c = p(Y=c) * p(x | Y=c)
post_x = s_x / s_x.sum()
print("scores [s_0, s_1]:", s_x)
print("p(Y | x)         :", post_x)
check("s_1", s_x[1], 0.1333)
check("s_0", s_x[0], 0.0136)
check("p(Y=1 | x)", post_x[1], 0.907)

### 3.4 Problem: zero probabilities
*Slide: Problem: Zero Probabilities.* All 5 disease patients had fatigue, so $\hat p(\text{Fatigue}=0\mid Y=1) = 0$.
Classify another patient $x = (1, 1, 0)$.

In [ ]:
x_new2 = [1, 1, 0]                                  # Fever=1, Cough=1, Fatigue=0
lik_x2 = ...  # TODO
s_x2 = ...  # TODO
print("scores [s_0, s_1]:", s_x2, "  p(Y | x):", s_x2 / s_x2.sum())
check("s_1 is vetoed", s_x2[1], 0.0)
check("s_0", s_x2[0], 0.0850)

**Q6.** Fever and cough are both present. Is $p(Y=1\mid x) = 0$ a sensible answer?

> **Your answer:** _..._

### 3.5 Laplace (add-$\alpha$) smoothing
*Slides: Laplace Smoothing · Example with Laplace Smoothing*

$$\hat p_\alpha(X_i = 1\mid Y=c) = \frac{\text{count}(X_i=1, Y=c) + \alpha}{N_c + \alpha K}, \qquad K = 2 \text{ for binary features}$$

The class priors are **not** smoothed (they stay $7/12$ and $5/12$).

In [ ]:
def fit_bernoulli_nb(df, features, target, alpha=1.0):
    """Return (prior, theta) where theta[c, i] = p(X_i = 1 | Y = c) with add-alpha smoothing."""
    N_c = df.groupby(target).size()
    counts = df.groupby(target)[features].sum()
    prior = N_c / N_c.sum()
    theta = ...  # TODO: (count + alpha) / (N_c + alpha * K), K = 2
    return prior, theta


prior_s, theta_s = fit_bernoulli_nb(df_train, features, "Disease", alpha=1)
display(theta_s)
check("theta (smoothed)", theta_s, [[3 / 9, 6 / 9, 3 / 9], [5 / 7, 4 / 7, 6 / 7]])

In [ ]:
rows = []
for x in ([1, 1, 0], [1, 0, 1]):
    p_mle = bayes(prior_y, bernoulli_likelihood(x, theta))[1]
    p_smooth = ...  # TODO: same as p_mle but with the smoothed parameters
    rows.append({"x": x, "p(Y=1|x) no smoothing": p_mle, "p(Y=1|x) Laplace": p_smooth})
smooth_table = pd.DataFrame(rows)
display(smooth_table)
check("x=(1,1,0) smoothed", smooth_table.iloc[0, 2], 0.219)
check("x=(1,0,1) smoothed", smooth_table.iloc[1, 2], 0.835)

### 3.6 How does $\alpha$ change the answer?

In [ ]:
alphas = np.logspace(-3, 3, 61)
p_110 = []
for a in alphas:
    pr, th = ...  # TODO: fit with this alpha
    p_110.append(bayes(pr, bernoulli_likelihood([1, 1, 0], th))[1])

plt.figure(figsize=(6, 3))
plt.semilogx(alphas, p_110)
plt.axhline(5 / 12, ls="--", c="gray", label="prior p(Y=1) = 5/12")
plt.axvline(1, ls=":", c="k", label=r"Laplace, $\alpha = 1$")
plt.xlabel(r"$\alpha$")
plt.ylabel("p(Y=1 | x=(1,1,0))")
plt.legend()
plt.show()

**Q7.** What happens when $\alpha \to 0$ and when $\alpha \to \infty$? Why does the curve approach the prior?

> **Your answer:** _..._

### 3.7 The same model in scikit-learn
*Slide: In Practice — "Libraries such as scikit-learn do this internally."*

In [ ]:
from sklearn.naive_bayes import BernoulliNB

X_train = df_train[features].to_numpy()
y_train = df_train["Disease"].to_numpy()
X_test = np.array([[1, 0, 1], [1, 1, 0]])

bnb = ...  # TODO: Laplace smoothing
...  # TODO: train on X_train, y_train

print("classes_        :", bnb.classes_)
print("p(Y=c)          :", np.exp(bnb.class_log_prior_))      # sklearn stores log-probabilities
print("p(X_i=1 | Y=c)  :\n", np.exp(bnb.feature_log_prob_))
print("p(Y | x)        :\n", bnb.predict_proba(X_test))
check("sklearn theta = our theta_s", np.exp(bnb.feature_log_prob_), theta_s)
check("sklearn p(Y=1|x)", bnb.predict_proba(X_test)[:, 1], [0.835, 0.219])

> Note: `bnb.class_prior` is the prior *you pass in* (default `None`); the fitted prior is
> `np.exp(bnb.class_log_prior_)`.

**What about $\alpha = 0$ (pure MLE) in scikit-learn?**

In [ ]:
bnb0 = BernoulliNB(alpha=0.0).fit(X_train, y_train)
print("alpha = 0     :", bnb0.predict_proba(X_test))

bnb_tiny = BernoulliNB(alpha=1e-10).fit(X_train, y_train)
print("alpha = 1e-10 :", bnb_tiny.predict_proba(X_test))
check("tiny alpha reproduces MLE", bnb_tiny.predict_proba(X_test)[:, 1], [0.907, 0.0])

**Q8.** Why does $\alpha = 0$ give `nan` (and a `divide by zero encountered in log` warning) instead of 0.907?

> **Your answer:** _..._

### 3.8 Work in log space
*Slide: In Practice: Work in Log Space.* A product of many small probabilities underflows to 0.

In [ ]:
p_words = np.full(500, 1e-3)            # a 500-word document, each word has probability 1e-3
product = ...  # TODO: multiply them
log_sum = ...  # TODO: add their logs instead
print("product     :", product)
print("sum of logs :", log_sum)
check("product underflows", product, 0.0)
check("sum of logs", log_sum, -3453.88, tol=0.01)

Now redo Naive Bayes in log space:
$\log s_c = \log p(Y=c) + \sum_i \log p(x_i\mid Y=c)$.
To turn log-scores into probabilities safely, use
$p(Y=c\mid x) = \exp\big(\log s_c - \text{logsumexp}(\log s)\big)$.

In [ ]:
def bernoulli_log_likelihood(x, theta):
    x = np.asarray(x)
    log_p = ...  # TODO: log p(X_i = x_i | Y = c)
    return ...  # TODO: product of probabilities -> sum of logs


log_s = np.log(prior_s.values) + bernoulli_log_likelihood(x_new, theta_s)
post_log = ...  # TODO: normalize in log space
print("log scores (ours)   :", log_s)
print("log scores (sklearn):", bnb.predict_joint_log_proba([x_new])[0])
print("p(Y | x)            :", post_log)
check("log scores = sklearn", log_s, bnb.predict_joint_log_proba([x_new])[0])
check("p(Y=1 | x)", post_log[1], 0.835)

**Back to Part 2.2:** with $k = 200$ repetitions (2400 flips) the direct computation breaks.
Redo it in log space.

In [ ]:
def log_likelihood(theta, n_heads, n_tails):
    return ...  # TODO: log of theta^#H (1-theta)^#T


k = 200
direct = bayes([0.9, 0.1], [likelihood(0.5, 8 * k, 4 * k), likelihood(0.8, 8 * k, 4 * k)])
log_s_coin = np.log([0.9, 0.1]) + np.array([log_likelihood(0.5, 8 * k, 4 * k),
                                            log_likelihood(0.8, 8 * k, 4 * k)])
post_k = ...  # TODO
print("direct    :", direct)
print("log space :", post_k)
check("p(bent | D), k=200", post_k[1], 1.0)

### 🧪 Practice 3 — A tiny spam filter (Bernoulli NB)
Each email is described by whether it **contains** the words *free*, *winner*, *meeting*.

1. Fit Bernoulli NB with $\alpha = 1$ (reuse `fit_bernoulli_nb`) and compute
   $p(\text{spam}\mid \text{free}=1, \text{winner}=0, \text{meeting}=1)$.
2. Check your answer against `BernoulliNB`.
3. Fit again with $\alpha = 0$ and classify $x = (0, 1, 1)$. What happens? Why?

In [ ]:
df_email = pd.DataFrame({
    "Spam":    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    "free":    [1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "winner":  [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    "meeting": [0, 0, 0, 0, 1, 1, 1, 0, 1, 0],
})
words = ["free", "winner", "meeting"]
df_email

In [ ]:
# 1. by hand
prior_e, theta_e = ...  # TODO: alpha = 1
x_email = [1, 0, 1]
post_e = ...  # TODO: posterior [ham, spam]
print(f"p(spam | x) = {post_e[1]:.3f}")

# 2. scikit-learn
bnb_e = ...  # TODO
print("sklearn     :", bnb_e.predict_proba([x_email])[0])
check("matches sklearn", post_e, bnb_e.predict_proba([x_email])[0])

In [ ]:
# 3. alpha = 0 and x = (0, 1, 1)
prior_0, theta_0 = fit_bernoulli_nb(df_email, words, "Spam", alpha=0)
s_0 = ...  # TODO: scores [ham, spam]
print("scores with alpha=0:", s_0, "  posterior:", s_0 / s_0.sum())

prior_1, theta_1 = fit_bernoulli_nb(df_email, words, "Spam", alpha=1)
print("posterior with alpha=1:", bayes(prior_1, bernoulli_likelihood([0, 1, 1], theta_1)))

**Q9.** Explain the result for $\alpha = 0$.

> **Your answer:** _..._

---
# Part 4 — Naive Bayes Variants
Pick the distribution of each feature (given the class) to match the type of data.

## 4.1 Categorical Naive Bayes
*Slide: Naive Bayes Variants: Discrete Features.* (New example.)

Predict whether a student is **late** for class ($Y=1$) from *Weather* $\in$ {sunny, cloudy, rainy}
and *Transport* $\in$ {bus, car, walk}.

$$\hat\theta_{i,k|c} = \frac{\text{count}(X_i = k, Y = c) + \alpha}{N_c + \alpha K_i}$$

In [ ]:
df_late = pd.DataFrame({
    "Late":      [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    "Weather":   ["rainy", "rainy", "rainy", "cloudy", "sunny",
                  "sunny", "sunny", "sunny", "cloudy", "cloudy", "rainy", "sunny"],
    "Transport": ["bus", "bus", "car", "bus", "bus",
                  "walk", "car", "bus", "walk", "car", "walk", "walk"],
})
pd.crosstab(df_late["Late"], df_late["Transport"])      # counts per class and value

In [ ]:
def categorical_table(df, feature, target, alpha=1.0):
    """p(X_i = k | Y = c): one row per class c, one column per value k."""
    counts = pd.crosstab(df[target], df[feature])      # count(X_i = k, Y = c)
    N_c = counts.sum(axis=1)
    K = ...  # TODO: number of possible values of this feature
    return ...  # TODO: (count + alpha) / (N_c + alpha * K)


tables = {f: categorical_table(df_late, f, "Late") for f in ["Weather", "Transport"]}
display(tables["Weather"], tables["Transport"])
check("p(walk | late)", tables["Transport"].loc[1, "walk"], 1 / 8)

In [ ]:
prior_late = df_late["Late"].value_counts(normalize=True).sort_index()
x_cat = {"Weather": "rainy", "Transport": "walk"}
lik_cat = ...  # TODO: p(rainy | c) * p(walk | c), one value per class
post_late = bayes(prior_late, lik_cat)
print(f"p(late | rainy, walk) = {post_late[1]:.3f}")
check("p(late | rainy, walk)", post_late[1], 0.309)

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()                              # categories -> 0, 1, 2 (alphabetical)
X_late = enc.fit_transform(df_late[["Weather", "Transport"]])
catnb = ...  # TODO
print("categories:", enc.categories_)

x_cat_enc = enc.transform(pd.DataFrame([x_cat]))
print("sklearn p(Y | rainy, walk):", catnb.predict_proba(x_cat_enc)[0])
check("sklearn agrees", catnb.predict_proba(x_cat_enc)[0], post_late)

**Your turn:** compute $p(\text{late}\mid \text{sunny}, \text{car})$ by hand and check it with `catnb`.

In [ ]:
x_cat2 = {"Weather": "sunny", "Transport": "car"}
post_late2 = ...  # TODO
print(f"p(late | sunny, car) = {post_late2[1]:.3f}")
check("matches CategoricalNB", post_late2, catnb.predict_proba(enc.transform(pd.DataFrame([x_cat2])))[0])

**Q10.** Why not encode sunny = 0, cloudy = 1, rainy = 2 and use Gaussian NB instead?

> **Your answer:** _..._

## 4.2 Gaussian Naive Bayes
*Slides: Gaussian Naive Bayes · Example: Gaussian Naive Bayes*

Predict flu ($Y=1$) from temperature $X_1$ and heart rate $X_2$:
$X_i\mid Y=c \sim \mathcal N(\mu_{i|c}, \sigma^2_{i|c})$.

> ⚠️ In `scipy.stats.norm.pdf(x, loc, scale)`, `scale` is the **standard deviation** $\sigma$, not $\sigma^2$.

In [ ]:
params = pd.DataFrame({
    "prior":   [0.7, 0.3],
    "mu_temp": [37.0, 38.5], "sd_temp": [0.5, 0.5],
    "mu_hr":   [75.0, 95.0], "sd_hr":   [10.0, 10.0],
}, index=pd.Index([0, 1], name="Flu"))
params

In [ ]:
x_temp, x_hr = 38.0, 80
dens_temp = ...  # TODO: N(38.0; mu, sd^2) for both classes
dens_hr = ...  # TODO: N(80; mu, sd^2) for both classes
s_flu = ...  # TODO: s_c = p(Y=c) * p(x1 | c) * p(x2 | c)
post_flu = s_flu / s_flu.sum()
print("density temp :", dens_temp)
print("density HR   :", dens_hr)
print("scores       :", s_flu)
print("p(Y | x)     :", post_flu)
check("N(38.0; 38.5, 0.5^2)", dens_temp[1], 0.484)
check("N(80; 75, 10^2)", dens_hr[0], 0.0352)
check("p(flu | x)", post_flu[1], 0.41, tol=0.01)

**Q11.** `norm.pdf(38.5, 38.5, 0.1)` is about 3.99. How can a "probability" be larger than 1? Is it a problem for Naive Bayes?

> **Your answer:** _..._

### A generative model can generate data
*Slide: Generative vs Discriminative — "can generate new examples $x \sim p(X\mid Y=c)$".*
Sample 2000 patients from the model above, then let `GaussianNB` learn the parameters back.

In [ ]:
N_flu = 2000
mu = params[["mu_temp", "mu_hr"]].to_numpy()        # shape (2 classes, 2 features)
sd = params[["sd_temp", "sd_hr"]].to_numpy()
y_flu = ...  # TODO: step 1: Y ~ Bernoulli(0.3)
X_flu = ...  # TODO: step 2: each row X | Y=y ~ N(mu[y], sd[y]^2), features independent

plt.figure(figsize=(5, 4))
for c, color in [(0, "tab:blue"), (1, "tab:red")]:
    plt.scatter(*X_flu[y_flu == c].T, s=5, alpha=0.4, c=color, label=["no flu", "flu"][c])
plt.scatter([x_temp], [x_hr], marker="*", s=250, c="k", label="new patient")
plt.xlabel("temperature (°C)")
plt.ylabel("heart rate (beats/min)")
plt.legend()
plt.show()

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb = ...  # TODO: fit GaussianNB on the sampled data
print("learned prior:", gnb.class_prior_)
print("learned mu   :\n", gnb.theta_)
print("learned sd   :\n", np.sqrt(gnb.var_))
print("p(Y | 38.0, 80):", gnb.predict_proba([[x_temp, x_hr]])[0])
check("learned mu ≈ true mu", gnb.theta_, mu, tol=1.0)
check("p(flu | x) ≈ 0.41", gnb.predict_proba([[x_temp, x_hr]])[0, 1], 0.41, tol=0.05)

In [ ]:
tt, hh = np.meshgrid(np.linspace(35.5, 40, 200), np.linspace(40, 130, 200))
P = gnb.predict_proba(np.column_stack([tt.ravel(), hh.ravel()]))[:, 1].reshape(tt.shape)

plt.figure(figsize=(5, 4))
cs = plt.contourf(tt, hh, P, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.6)
plt.contour(tt, hh, P, levels=[0.5], colors="k")
plt.colorbar(cs, label="p(flu | x)")
plt.scatter([x_temp], [x_hr], marker="*", s=250, c="k")
plt.xlabel("temperature (°C)")
plt.ylabel("heart rate (beats/min)")
plt.title("Gaussian NB decision boundary (black)")
plt.show()

**Q12 (optional).** The boundary is (almost) a straight line. Why? *Hint: compare $\sigma$ between the two classes.*

> **Your answer:** _..._

## 4.3 Multinomial Naive Bayes
*Slides: Multinomial Naive Bayes · Example · Estimates · Classify*

Documents → bag of words (counts). $\hat\theta_{w|c} = \dfrac{\text{count}(w,c)+\alpha}{\sum_{w'}\text{count}(w',c)+\alpha d}$,
$\;s_c = p(Y=c)\prod_w \theta_{w|c}^{x_w}$.

The four training documents below add up to the word counts on the slide.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "apple apple banana orange",            # Topic A
    "apple apple banana banana orange",     # Topic A
    "banana banana banana orange",          # Topic B
    "apple banana banana orange",           # Topic B
]
y_doc = np.array([1, 1, 0, 0])              # 1 = Topic A, 0 = Topic B

vec = CountVectorizer()
X_doc = vec.fit_transform(docs).toarray()   # one row per document, one column per word
vocab = vec.get_feature_names_out()
pd.DataFrame(X_doc, columns=vocab)

In [ ]:
count_A = ...  # TODO: total count of each word over the Topic A documents
count_B = ...  # TODO: same for Topic B
word_counts = pd.DataFrame({"Topic A": count_A, "Topic B": count_B}, index=vocab)
word_counts.loc["Total words"] = word_counts.sum()
display(word_counts)
check("Topic A counts", count_A, [4, 3, 2])
check("Topic B counts", count_B, [1, 5, 2])

In [ ]:
alpha, d = 1, len(vocab)
theta_A = ...  # TODO: smoothed word probabilities for Topic A
theta_B = ...  # TODO: same for Topic B
print("theta_A:", theta_A, "  theta_B:", theta_B)
check("theta_A", theta_A, [5 / 12, 4 / 12, 3 / 12])
check("theta_B", theta_B, [2 / 11, 6 / 11, 3 / 11])

In [ ]:
x_doc = vec.transform(["apple orange orange"]).toarray()[0]     # apple=1, banana=0, orange=2
s_A = ...  # TODO: p(A) * prod_w theta_w^x_w   (equal priors: 0.5)
s_B = ...  # TODO
post_A = s_A / (s_A + s_B)
print(f"x = {x_doc},  s_A = {s_A:.4f},  s_B = {s_B:.4f},  p(Topic A | doc) = {post_A:.3f}")
check("s_A", s_A, 45 / 3456, tol=1e-5)
check("s_B", s_B, 18 / 2662, tol=1e-5)
check("p(Topic A | doc)", post_A, 0.658)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

mnb = ...  # TODO
print("feature_count_ (rows: B, A):\n", mnb.feature_count_)
print("p(w | c):\n", np.exp(mnb.feature_log_prob_))
print("p(Y | doc):", mnb.predict_proba([x_doc])[0])
check("sklearn p(Topic A | doc)", mnb.predict_proba([x_doc])[0, 1], post_A)

> scikit-learn's prior is (number of documents in $c$) / (total documents) = 2/4 = 0.5,
> the same simplification as on the slide.

## 4.4 Complement Naive Bayes (optional)
*Slide: Complement Naive Bayes.* For each class $c$, estimate the word distribution of its
**complement** $\neg c$, then choose the class whose complement fits the document **worst**:
$\hat y = \arg\max_c \; -\sum_w x_w \log\hat\theta_{w|\neg c}$.

In [ ]:
theta_notA = theta_B                    # with 2 classes, the complement of A is B
theta_notB = theta_A
score_A = ...  # TODO: -sum_w x_w log theta_{w | not A}
score_B = ...  # TODO
print(f"CNB scores: A = {score_A:.3f}, B = {score_B:.3f}  ->", "Topic A" if score_A > score_B else "Topic B")

from sklearn.naive_bayes import ComplementNB
compnb = ComplementNB(alpha=1).fit(X_doc, y_doc)
print("ComplementNB  p(Y | doc):", compnb.predict_proba([x_doc])[0])
print("MultinomialNB p(Y | doc):", mnb.predict_proba([x_doc])[0])
check("sklearn stores -log theta_not_c", compnb.feature_log_prob_, -np.log([theta_notB, theta_notA]))

**Q13.** Why do ComplementNB and MultinomialNB give exactly the same answer here?

> **Your answer:** _..._

### 🧪 Practice 4 — SMS spam with Multinomial NB
1. Turn the messages into word counts with `CountVectorizer` and fit `MultinomialNB(alpha=1)`.
2. Compute $p(\text{spam})$ for the three new messages.
3. List the 5 most "spammy" words, i.e. the largest $\log\hat\theta_{w|\text{spam}} - \log\hat\theta_{w|\text{ham}}$.

In [ ]:
sms = [
    ("win a free prize now", 1),
    ("free entry win cash now", 1),
    ("claim your free prize call now", 1),
    ("urgent you win cash claim now", 1),
    ("free free free call now", 1),
    ("are we meeting for lunch today", 0),
    ("call me when you get home", 0),
    ("lunch at noon see you there", 0),
    ("can you send the report today", 0),
    ("see you at the meeting", 0),
    ("i will call you tonight", 0),
    ("thanks for lunch today", 0),
]
sms_text = [t for t, _ in sms]
sms_y = np.array([label for _, label in sms])      # 1 = spam, 0 = ham
new_sms = ["free lunch today", "call now to claim your prize", "see you tonight"]

In [ ]:
vec_sms = CountVectorizer()
X_sms = ...  # TODO: learn the vocabulary and count the words
mnb_sms = ...  # TODO
p_spam = ...  # TODO: p(spam) for new_sms
display(pd.DataFrame({"message": new_sms, "p(spam)": p_spam}))

log_ratio = ...  # TODO: spam row minus ham row
top5 = vec_sms.get_feature_names_out()[np.argsort(log_ratio)[::-1][:5]]
print("most spammy words:", top5)

**Q14.** (a) What does `CountVectorizer` do with a word it never saw in training (e.g. *to* in "call now to claim your prize")? (b) *lunch* never appears in spam. What would $\alpha = 0$ predict for "free lunch today"?

> **Your answer:** _..._